# Step 1 - Baseline reproduction

Reproduce the XGBoost baseline with random undersampling, seed 42, sampling ratio 0.1 and decision threshold 0.5. The recorded run achieved validation average precision of 0.705831; the original modelling notebook reported 0.701045. The discrepancy is retained in the comparison table rather than treated as an exact reproduction.

Model fitting uses the training split. Validation provides development metrics. Test data are inspected only for structural checks in this step.


Displayed outputs are retained from the recorded experiment. Documentation and path configuration have been updated for the packaged layout; the model calculations have not been rerun during packaging.


## 1. Environment and configuration

Use the package versions in `requirements-tuning.txt`. Define imports and helper functions before the path configuration cell. `PROJECT_DIR` identifies the directory containing the notebooks and output folders; `DATA_DIR` identifies the directory containing the three engineered CSV splits. Local Jupyter is the default, with optional Colab configuration in the path cell.


In [ ]:
# Install only if needed, in the active notebook kernel:
# %pip install numpy pandas scikit-learn imbalanced-learn xgboost joblib

In [ ]:
from pathlib import Path
import argparse
import gc
import hashlib
import importlib.metadata
import json
import platform
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             precision_score, recall_score, f1_score,
                             confusion_matrix)

SEED = 42
LABEL = 'isFraud'
THRESHOLD = 0.5
SAMPLING_RATIO = 0.1
EXCLUDE = [LABEL, 'TransactionID', 'TransactionDT', 'TransactionDateTime',
           'Transaction_Day', 'uid']
EXPECTED = {'Average precision': 0.701045, 'ROC AUC': 0.947089,
            'Precision': 0.751929, 'Recall': 0.590951, 'F1': 0.661792,
            'False positive rate': 0.007079,
            'True positives': 1267, 'False positives': 418,
            'False negatives': 877, 'True negatives': 58634}


def json_save(path, value):
    Path(path).write_text(json.dumps(value, indent=2, default=str), encoding='utf-8')


def versions():
    result = {'python': platform.python_version()}
    for name in ['numpy', 'pandas', 'scikit-learn', 'imbalanced-learn', 'xgboost', 'joblib']:
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = 'NOT INSTALLED'
    return result


def locate_files(data_dir):
    data_dir = Path(data_dir)
    aliases = {'train': ['train.csv', 'train (2).csv'],
               'val': ['val.csv'], 'test': ['test.csv', 'test (3).csv']}
    paths = {}
    for split, names in aliases.items():
        matches = [data_dir / name for name in names if (data_dir / name).is_file()]
        if len(matches) != 1:
            raise ValueError(f'Expected exactly one of {names} in {data_dir}; found {matches}')
        paths[split] = matches[0]
    return paths


In [ ]:
# Dataset and output paths.

# Project root: directory containing these notebooks and output folders.
PROJECT_DIR = Path.cwd()

# Optional Google Colab setup:
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = Path('/content/drive/MyDrive/fine tuning')

# CSV splits can be stored in data/ or directly in PROJECT_DIR.
DATA_DIR = PROJECT_DIR / 'data' if (PROJECT_DIR / 'data').is_dir() else PROJECT_DIR
OUTPUT_DIR = PROJECT_DIR / 'baseline_outputs'
paths = locate_files(DATA_DIR)
print(versions())

{'python': '3.13.16', 'numpy': '2.1.3', 'pandas': '2.2.3', 'scikit-learn': '1.6.1', 'imbalanced-learn': '0.14.2', 'xgboost': '3.4.1', 'joblib': '1.6.0'}


## 2. Dataset checks

The training, validation and test splits contain 440,018, 61,196 and 89,326 rows respectively, with 488 columns each. Verify binary labels, unique transaction IDs, non-overlapping splits, consistent column order and chronological ordering. Test-label inspection is limited to the structural summary.


In [ ]:
def audit_data(paths, output_dir):
    """Structural checks only; never fit or score on test data."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    summaries, headers, ids, fingerprints = {}, {}, {}, {}
    for split, path in paths.items():
        headers[split] = pd.read_csv(path, nrows=0).columns.tolist()
        required = set(EXCLUDE)
        if not required.issubset(headers[split]):
            raise ValueError(f'{split}: missing columns {required - set(headers[split])}')
        data = pd.read_csv(path, usecols=['TransactionID', 'TransactionDT', LABEL])
        assert data[LABEL].notna().all() and data[LABEL].isin([0, 1]).all(), f'{split}: invalid labels'
        assert data['TransactionID'].notna().all(), f'{split}: missing IDs'
        assert not data['TransactionID'].duplicated().any(), f'{split}: duplicate IDs'
        assert data['TransactionDT'].notna().all(), f'{split}: missing times'
        assert data['TransactionDT'].is_monotonic_increasing, f'{split}: time order changed'
        ids[split] = set(data['TransactionID'])
        summaries[split] = dict(rows=len(data), columns=len(headers[split]),
            fraud_count=int(data[LABEL].sum()), fraud_rate=float(data[LABEL].mean()),
            missing_labels=int(data[LABEL].isna().sum()), duplicate_ids=0,
            time_min=float(data['TransactionDT'].min()), time_max=float(data['TransactionDT'].max()))
        digest = hashlib.sha256()
        with path.open('rb') as file:
            for chunk in iter(lambda: file.read(8 * 1024 * 1024), b''):
                digest.update(chunk)
        fingerprints[split] = {'filename': path.name, 'sha256': digest.hexdigest()}
    checks = {
        'identical_column_order': headers['train'] == headers['val'] == headers['test'],
        'disjoint_transaction_ids': all(not ids[a].intersection(ids[b]) for a, b in
                                       [('train', 'val'), ('train', 'test'), ('val', 'test')]),
        'strict_chronological_splits': summaries['train']['time_max'] < summaries['val']['time_min']
                                   and summaries['val']['time_max'] < summaries['test']['time_min']}
    assert all(checks.values()), checks
    report = {'splits': summaries, 'checks': checks, 'input_files': fingerprints,
              'test_performance_evaluated': False}
    json_save(output_dir / 'data_audit.json', report)
    table = pd.DataFrame(summaries).T
    print(table.to_string())
    print('Checks:', checks)
    return report



audit = audit_data(paths, OUTPUT_DIR)


           rows  columns  fraud_count  fraud_rate  missing_labels  duplicate_ids    time_min    time_max
train  440018.0    488.0      15405.0    0.035010             0.0            0.0     86400.0  11145595.0
val     61196.0    488.0       2144.0    0.035035             0.0            0.0  11145618.0  13132788.0
test    89326.0    488.0       3114.0    0.034861             0.0            0.0  13132810.0  15811131.0
Checks: {'identical_column_order': True, 'disjoint_transaction_ids': True, 'strict_chronological_splits': True}


## 3. Preprocessing

Exclude the six identifier, label and time-helper columns. Preserve the order of the remaining 482 features, replace infinite values with NaN, and fit the categorical encoder on training data only. The recorded run contains 37 categorical columns. No additional scaling or imputation is applied.


In [ ]:
def prepare_data(paths):
    """Match Modelling.ipynb: preserve column order and string-cast categories."""
    header = pd.read_csv(paths['train'], nrows=0).columns.tolist()
    features = [column for column in header if column not in EXCLUDE]
    prepared, targets = {}, {}
    categorical_columns, encoder = None, None
    for split in ['train', 'val']:
        frame = pd.read_csv(paths[split], usecols=[LABEL] + features, low_memory=False)
        targets[split] = frame.pop(LABEL).astype(int).to_numpy()
        assert frame.columns.tolist() == features
        frame.replace([np.inf, -np.inf], np.nan, inplace=True)
        if split == 'train':
            categorical_columns = frame.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
            if categorical_columns:
                encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
                frame[categorical_columns] = encoder.fit_transform(frame[categorical_columns].astype(str))
        elif categorical_columns:
            frame[categorical_columns] = encoder.transform(frame[categorical_columns].astype(str))
        prepared[split] = frame
    print(f'Prepared {len(features)} model features; {len(categorical_columns)} categorical columns.')
    print('Training shape:', prepared['train'].shape, 'Validation shape:', prepared['val'].shape)
    return prepared, targets, features, categorical_columns, encoder



prepared, targets, features, categorical_columns, encoder = prepare_data(paths)


Prepared 482 model features; 37 categorical columns.
Training shape: (440018, 482) Validation shape: (61196, 482)


## 4. Model fitting and baseline comparison

Fit XGBoost using the original constructor and undersampling settings. Compare validation metrics with the original recorded values and save the fitted artifacts. No hyperparameter search, early stopping or threshold optimization is performed.


In [ ]:
def transform_for_prediction(raw_frame, bundle):
    """Apply a saved bundle to already-engineered rows with the same features."""
    missing = set(bundle['features']) - set(raw_frame.columns)
    if missing:
        raise ValueError(f'Missing engineered features: {sorted(missing)}')
    frame = raw_frame.loc[:, bundle['features']].copy()
    frame.replace([np.inf, -np.inf], np.nan, inplace=True)
    columns = bundle['categorical_columns']
    if columns:
        frame[columns] = bundle['encoder'].transform(frame[columns].astype(str))
    return frame


def train_and_evaluate(prepared, targets, features, categorical_columns, encoder, output_dir):
    from imblearn.under_sampling import RandomUnderSampler
    from xgboost import XGBClassifier

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    sampler = RandomUnderSampler(sampling_strategy=SAMPLING_RATIO, random_state=SEED)
    started = time.perf_counter()
    X_res, y_res = sampler.fit_resample(prepared['train'], targets['train'])
    counts = {str(int(k)): int(v) for k, v in zip(*np.unique(y_res, return_counts=True))}
    print('Training counts after undersampling:', counts)
    assert counts == {'0': 154050, '1': 15405}, 'Training data differs from uploaded baseline.'
    # Match the original constructor exactly; remaining defaults are version-dependent.
    model = XGBClassifier(random_state=SEED, scale_pos_weight=1.0,
                          eval_metric='logloss', n_jobs=-1)
    model.fit(X_res, y_res)
    fit_seconds = time.perf_counter() - started
    del X_res, y_res
    gc.collect()
    probabilities = model.predict_proba(prepared['val'])[:, 1]
    predictions = model.predict(prepared['val'])  # Original notebook's default decision rule.
    assert np.array_equal(predictions, (probabilities > THRESHOLD).astype(int))
    y = targets['val']
    tn, fp, fn, tp = confusion_matrix(y, predictions, labels=[0, 1]).ravel()
    actual = {
        'Average precision': float(average_precision_score(y, probabilities)),
        'ROC AUC': float(roc_auc_score(y, probabilities)),
        'Precision': float(precision_score(y, predictions, zero_division=0)),
        'Recall': float(recall_score(y, predictions, zero_division=0)),
        'F1': float(f1_score(y, predictions, zero_division=0)),
        'False positive rate': float(fp / (fp + tn)),
        'True positives': int(tp), 'False positives': int(fp),
        'False negatives': int(fn), 'True negatives': int(tn)}
    comparison = pd.DataFrame({'Original notebook': EXPECTED, 'This run': actual})
    comparison['Difference'] = comparison['This run'] - comparison['Original notebook']
    comparison['Matches recorded rounding'] = np.isclose(comparison['This run'],
        comparison['Original notebook'], atol=0.000005, rtol=0)
    print(comparison.to_string())
    print(f'Training including sampling: {fit_seconds:.1f} seconds')
    comparison.to_csv(output_dir / 'baseline_comparison.csv', index_label='Metric')
    metadata = {'status': 'trained_and_validated', 'seed': SEED, 'sampling_ratio': SAMPLING_RATIO,
        'threshold': THRESHOLD, 'threshold_rule': 'score > 0.5',
        'versions': versions(), 'constructor_parameters': model.get_params(),
        'resampled_class_counts': counts, 'training_seconds_including_sampling': fit_seconds,
        'validation_metrics': actual, 'test_performance_evaluated': False,
        'feature_count': len(features), 'categorical_count': len(categorical_columns),
        'validation_has_already_been_used_for_model_selection': True}
    json_save(output_dir / 'baseline_run.json', metadata)
    model.save_model(output_dir / 'xgboost_rus_baseline.json')
    (output_dir / 'xgboost_effective_config.json').write_text(model.get_booster().save_config())
    bundle = {'model': model, 'encoder': encoder, 'features': features,
        'categorical_columns': categorical_columns, 'exclude': EXCLUDE,
        'threshold': THRESHOLD, 'threshold_rule': '>', 'metadata': metadata}
    bundle_path = output_dir / 'baseline_bundle.joblib'
    joblib.dump(bundle, bundle_path, compress=3)
    # Check that the saved model reproduces the in-memory predictions.
    reloaded = joblib.load(bundle_path)
    probe = prepared['val'].iloc[:1000]
    np.testing.assert_allclose(reloaded['model'].predict_proba(probe)[:, 1],
                               probabilities[:len(probe)], rtol=0, atol=0)
    return model, comparison, bundle



model, comparison, bundle = train_and_evaluate(
    prepared, targets, features, categorical_columns, encoder, OUTPUT_DIR)
comparison


Training counts after undersampling: {'0': 154050, '1': 15405}
                     Original notebook      This run  Difference  Matches recorded rounding
Average precision             0.701045      0.705831    0.004786                      False
ROC AUC                       0.947089      0.947211    0.000122                      False
Precision                     0.751929      0.740290   -0.011639                      False
Recall                        0.590951      0.595616    0.004665                      False
F1                            0.661792      0.660119   -0.001673                      False
False positive rate           0.007079      0.007587    0.000508                      False
True positives             1267.000000   1277.000000   10.000000                      False
False positives             418.000000    448.000000   30.000000                      False
False negatives             877.000000    867.000000  -10.000000                      False
True negatives   

,Original notebook,This run,Difference,Matches recorded rounding
Average precision,0.701045,0.705831,0.004786,False
ROC AUC,0.947089,0.947211,0.000122,False
Precision,0.751929,0.740290,-0.011639,False
Recall,0.590951,0.595616,0.004665,False
F1,0.661792,0.660119,-0.001673,False
False positive rate,0.007079,0.007587,0.000508,False
True positives,1267.000000,1277.000000,10.000000,False
False positives,418.000000,448.000000,30.000000,False
False negatives,877.000000,867.000000,-10.000000,False
True negatives,58634.000000,58604.000000,-30.000000,False


## 5. Saved-model verification

Reload the model bundle and verify predictions on engineered validation rows. The bundle contains the fitted model, categorical encoder, feature order and threshold. Raw-data feature engineering remains a separate preprocessing stage.


In [ ]:
raw_probe = pd.read_csv(paths['val'], nrows=1000, low_memory=False)
loaded = joblib.load(OUTPUT_DIR / 'baseline_bundle.joblib')
probe = transform_for_prediction(raw_probe, loaded)
np.testing.assert_allclose(
    loaded['model'].predict_proba(probe),
    model.predict_proba(prepared['val'].iloc[:len(raw_probe)]),
    rtol=0, atol=0)
print('Reload check passed. Test performance was not evaluated.')


Reload check passed. Test performance was not evaluated.
